We downsample to 8000 hz to reduce resource usage

In [1]:
from os import listdir
from imports import *
import librosa as lb 
import numpy as np

sample_rate = 8000
fedez_files = list(map(lambda x : "fedez/" + x, listdir("fedez")))
fibra_files = list(map(lambda x : "fibra/" + x, listdir("fibra")))

FileNotFoundError: [Errno 2] No such file or directory: 'fedez'

In [ ]:
def ts_from_file(filename, rate = sample_rate):
    return lb.load(filename, sr = rate)[0]

def load_files(x, rate = sample_rate):
    return list(map(lambda y: lb.load(y, sr = rate)[0], x))

def make_2d(x: np.ndarray):
    return np.reshape(x, (np.shape(x)[0], 1))

def load_and_reshape(x, rate = sample_rate):
    return list(map(lambda y: make_2d(lb.load(y, sr = rate)[0]), x))


In [ ]:
sample_fedez = ts_from_file(fedez_files[10])
lb.display.waveshow(sample_fedez, sr = sample_rate)

In [ ]:
lb.display.waveshow(sample_fedez[sample_rate * 20:sample_rate*21], sr = sample_rate)

Since using waveforms directly would be computationally too expensive, we will use features such as onset and chroma.

In [ ]:
from math import ceil
from random import randint

def extract_chroma_clips(chroma, n_samples = 10, sample_len = 80) :
    chroma /= np.max(np.abs(chroma))
    n = np.shape(chroma)[1]
    slices_start = [randint(0, n - sample_len) for _ in range(n_samples)]
    return [np.transpose(chroma[0:12,x:x + sample_len]) for x in slices_start]

def chroma_from_file(filename):
    ts = lb.load(filename, sr = sample_rate)[0]
    return lb.feature.chroma_stft(y=ts, sr = sample_rate)

def chroma_clips_from_file(filename, samples = 11):
    return extract_chroma_clips(chroma_from_file(filename), n_samples=samples)

def mfcc_from_file(filename):
    ts = lb.load(filename, sr = sample_rate)[0]
    return lb.feature.mfcc(y=ts, sr = sample_rate)

def extract_mfcc_clips(mfcc, n_samples = 10, sample_len = 80) :
    mfcc /= np.max(np.abs(mfcc))
    n = np.shape(mfcc)[1]
    slices_start = [randint(0, n - sample_len) for _ in range(n_samples)]
    return [np.transpose(mfcc[0:20,x:x + sample_len]) for x in slices_start]

def mfcc_clips_from_file(filename, samples = 11):
    return extract_mfcc_clips(mfcc_from_file(filename), n_samples=samples)

def onset_from_file(filename, hl = 512, sr = sample_rate):
    odf = lb.onset.onset_strength(y=lb.load(filename)[0], hop_length=hl, sr = sr)
    return (odf - odf.mean()) / odf.std()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.transforms as mpt
fig, ax = plt.subplots()
trans = mpt.blended_transform_factory(ax.transData, ax.transAxes)
chroma_sample = chroma_from_file(fedez_files[0])
lb.display.specshow(chroma_sample, y_axis='chroma', x_axis='time', ax=ax,sr =sample_rate)

Since we are working with a lot of data, we will need to accept some compromises while dealing with clustering using time series directly. DTW would be the preferred option, but is quadratic in both the number of time series and their length. Therefore it is almost impossible to use it on the raw soundwaves(at least on my laptop). To avoid this issue we will use two sets of features(chroma and mfcc) that allow to carry information through a shorter, multi-dimensional (32D in this case) time series.
We will also try other approaches, such as cutting the songs to a specific length.

In [ ]:
from tslearn import utils, clustering
from tslearn.metrics import dtw
def mfcc_file(filename, fps = 0.5):
    ts = lb.load(filename, sr = sample_rate)[0]
    mfcc = lb.feature.mfcc(y = ts, sr = sample_rate, hop_length=ceil(sample_rate/fps))
    mfcc /= np.abs(np.max(mfcc))
    return mfcc

def chroma_file(filename, fps=0.5):
    ts = lb.load(filename, sr = sample_rate)[0]
    chroma = lb.feature.chroma_stft(y = ts, sr = sample_rate, hop_length=ceil(sample_rate/fps))
    chroma /= np.abs(np.max(chroma))
    return chroma 

def k_clusters(dataset, num_clusters, metric):
    kmeans = clustering.TimeSeriesKMeans(
        n_clusters=num_clusters,
        metric=metric,
        random_state=0
    )
    kmeans.fit(dataset)
    silhouette =  clustering.silhouette_score(dataset, kmeans.labels_, metric=metric)
    _, counts = np.unique(sorted(kmeans.labels_), return_counts=True)
    print(num_clusters, "clusters: ", silhouette)
    print("Distribution: ", counts)


all_files = fedez_files + fibra_files

all_chroma = utils.to_time_series_dataset(list(map(lambda y: np.transpose(chroma_file(y)),all_files)))
all_mfcc = utils.to_time_series_dataset(list(map(lambda y: np.transpose(mfcc_file(y)),all_files)))


In [ ]:
kmeans = clustering.TimeSeriesKMeans(
    n_clusters=2,
    metric="dtw", 
    random_state=0,
    )
kmeans.fit(all_chroma)
chroma_labels = kmeans.labels_


In [ ]:
print(chroma_labels)

In [ ]:
for i in range(2, 11):
    k_clusters(all_mfcc, i, "dtw")

In [ ]:
chroma_silhouette = clustering.silhouette_score(all_chroma,chroma_labels, metric="dtw")
print("Chroma silhouette score: ", chroma_silhouette)

We will now also try to operate on raw soundwaves, trimming them to a fixed duration.

In [ ]:
def load_trimmed(filename, length, sr):
    ts = lb.load(filename, sr = sr, duration = length)[0]
    max = np.abs(np.max(ts))
    if max != 0:
        ts /= max
    return ts

all_ts_trimmed = utils.to_time_series_dataset([load_trimmed(x, 20, 200) for x in all_files])
print(np.shape(all_ts_trimmed[0]))
for i in range(2, 11):
    k_clusters(all_ts_trimmed, i, "euclidean")

In [ ]:
from tslearn.matrix_profile import MatrixProfile
from math import ceil
def compute_mp(ts, m = 22050*2):
    X = np.asarray(ts, dtype=np.float32).reshape(-1, 1)
    mp = MatrixProfile(m)
    return mp.fit_transform([X])[0].ravel()

def plot_matrix_profile(filename, sr=22050, hop_length=512, title=""):
    x = onset_from_file(filename, sr = sr, hl = hop_length)
    pps = sr / hop_length 
    m = ceil(pps * 2)
    x = np.asarray(x)
    profile = np.asarray(compute_mp(x, m))

    t_x = lb.frames_to_time(np.arange(len(x)), sr=sr, hop_length=hop_length)
    t_p = lb.frames_to_time(np.arange(len(profile)), sr=sr, hop_length=hop_length)

    plt.figure(figsize=(12, 6))

    plt.subplot(2, 1, 1)
    plt.plot(t_x, x)
    plt.title(title if title else "Onset Strength")
    plt.xlabel("Time (s)")

    plt.subplot(2, 1, 2)
    plt.plot(t_p, profile)
    plt.title("Matrix Profile")
    plt.xlabel("Time (s)")

    plt.tight_layout()
    plt.show()

plot_matrix_profile(fibra_files[10])


As we can see, there are some areas where the matrix profile is lower, which probably correspond to choruses. Listening to the song confirms this.

In order to try to classify songs by artists using shapelets, we first need to choose a feature and normalize it, then extract some short(~10s) clips from the songs and feed them to TSLearn's shapelet learning model. We will use a third of each author's songs as training data.

In [ ]:
from tslearn.shapelets import grabocka_params_to_shapelet_size_dict, LearningShapelets
from random import sample
from keras.optimizers import Adam
fedez_tr = sample(fedez_files, k = len(fedez_files)//3)
fibra_tr = sample(fibra_files, k = len(fibra_files)//3)
x_tr = []
y_tr = []
for song in fedez_tr:
    clips = chroma_clips_from_file(song)
    x_tr.extend(clips)
    y_tr.extend([0] * len(clips))

for song in fibra_tr:
    clips = chroma_clips_from_file(song)
    x_tr.extend(clips)
    y_tr.extend([1] * len(clips))


shp = LearningShapelets(
    n_shapelets_per_size={16: 10},
    optimizer=Adam(0.01),
    batch_size=32,
    max_iter=1000,
    random_state=0,
    verbose=0
)

shp.fit(x_tr, y_tr)

def predict_song(filename):
    clips = chroma_clips_from_file(filename, 11)
    tot = np.sum(shp.predict( np.stack(clips)))
    if tot < 6:
        return 0
    else:
        return 1



In [ ]:
results = list(map(predict_song, fedez_files))
results.extend(map(predict_song, fibra_files))
expectation = [0] * len(fedez_files) + [1] * len(fibra_files)
from sklearn.metrics import accuracy_score
print("accuracy:", accuracy_score(expectation, results))

Using chroma does not yields an accuracy of 0.72. We will try the same strategy using onset strength and MFCC.

In [ ]:
from tslearn.shapelets import grabocka_params_to_shapelet_size_dict, LearningShapelets
from random import sample
from keras.optimizers import Adam
fedez_tr = sample(fedez_files, k = len(fedez_files)//3)
fibra_tr = sample(fibra_files, k = len(fibra_files)//3)
x_tr = []
y_tr = []
for song in fedez_tr:
    clips = mfcc_clips_from_file(song)
    x_tr.extend(clips)
    y_tr.extend([0] * len(clips))

for song in fibra_tr:
    clips = mfcc_clips_from_file(song)
    x_tr.extend(clips)
    y_tr.extend([1] * len(clips))


mfcc_shp = LearningShapelets(
    n_shapelets_per_size={16: 20},
    optimizer=Adam(0.01),
    batch_size=32,
    max_iter=1000,
    random_state=0,
    verbose=0
)

mfcc_shp.fit(x_tr, y_tr)

def predict_song(filename):
    clips = mfcc_clips_from_file(filename, 11)
    tot = np.sum(mfcc_shp.predict( np.stack(clips)))
    if tot < 6:
        return 0
    else:
        return 1



In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
results = list(map(predict_song, fedez_files))
results.extend(map(predict_song, fibra_files))
expectation = [0] * len(fedez_files) + [1] * len(fibra_files)
ConfusionMatrixDisplay.from_predictions(y_true=expectation,y_pred=results, display_labels=["fedez", "fibra"], cmap="Blues")
print("MFCC accuracy:", accuracy_score(expectation, results))

In [ ]:
fedez_tr = sample(fedez_files, k = len(fedez_files)//3)
fibra_tr = sample(fibra_files, k = len(fibra_files)//3)
x_tr = []
y_tr = []
def extract_onset_clips(onset, n_samples = 10, sample_len = 200) :
    onset /= np.max(np.abs(onset))
    n = np.shape(onset)[0]
    onset.reshape(-1,1)
    slices_start = [randint(0, n - sample_len) for _ in range(n_samples)]
    return [onset[x:x + sample_len] for x in slices_start]

def onset_clips_from_file(file, n_samples = 11):
    return extract_onset_clips(onset_from_file(file), n_samples=n_samples)

for song in fedez_tr:
    clips = onset_clips_from_file(song)
    x_tr.extend(clips)
    y_tr.extend([0] * len(clips))

for song in fibra_tr:
    clips = onset_clips_from_file(song)
    x_tr.extend(clips)
    y_tr.extend([1] * len(clips))

shapelet_sizes = grabocka_params_to_shapelet_size_dict(
    n_ts=len(x_tr), ts_sz=200, n_classes=2, l=0.2, r=1
)

shp = LearningShapelets(
    n_shapelets_per_size=shapelet_sizes,
    optimizer=Adam(0.01),
    batch_size=32,
    max_iter=1000,
    random_state=0,
    verbose=0
)

shp.fit(x_tr, y_tr)

def predict_song(filename):
    clips = onset_clips_from_file(filename, 11)
    tot = np.sum(shp.predict( np.stack(clips)))
    if tot < 6:
        return 0
    else:
        return 1

results = list(map(predict_song, fedez_files))
results.extend(map(predict_song, fibra_files))
expectation = [0] * len(fedez_files) + [1] * len(fibra_files)
from sklearn.metrics import accuracy_score
print("Onset accuracy:", accuracy_score(expectation, results))